# Hands-On Lab: Optimizers in Detail (SGD, Momentum, RMSProp, Adam)
Accompanies **Chapter 6: Optimizers in Detail**.

In this lab, we:
1. Implement Gradient Descent with Momentum, RMSProp, and Adam from scratch.
2. Optimize a non-convex 2D loss surface with an anisotropic canyon.
3. Compare trajectory paths, oscillation damping, and convergence speed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define anisotropic quadratic loss: f(x, y) = 0.1 * x^2 + 2.0 * y^2 (Canyon shape)
def loss_func(x, y):
    return 0.1 * x**2 + 2.0 * y**2

def grad_func(x, y):
    return np.array([0.2 * x, 4.0 * y])

# Optimizer implementations
def run_optimizer(opt_name, lr=0.1, steps=50, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array([4.0, 3.0])  # Starting point
    path = [w.copy()]
    m = np.zeros(2)
    v = np.zeros(2)
    
    for t in range(1, steps + 1):
        g = grad_func(w[0], w[1])
        if opt_name == 'SGD':
            w = w - lr * g
        elif opt_name == 'Momentum':
            m = beta1 * m + lr * g
            w = w - m
        elif opt_name == 'RMSProp':
            v = beta1 * v + (1 - beta1) * (g**2)
            w = w - lr * g / (np.sqrt(v) + eps)
        elif opt_name == 'Adam':
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * (g**2)
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

# Run optimizers
path_sgd = run_optimizer('SGD', lr=0.35, steps=40)
path_mom = run_optimizer('Momentum', lr=0.15, steps=40, beta1=0.8)
path_adam = run_optimizer('Adam', lr=0.4, steps=40)

# Plot loss contours and optimizer trajectories
x_grid = np.linspace(-5, 5, 200)
y_grid = np.linspace(-4, 4, 200)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
Z_mesh = loss_func(X_mesh, Y_mesh)

plt.figure(figsize=(10, 6))
plt.contour(X_mesh, Y_mesh, Z_mesh, levels=np.logspace(-1, 2, 20), cmap='viridis', alpha=0.6)
plt.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', label='SGD (Oscillates across canyon)', linewidth=1.5)
plt.plot(path_mom[:, 0], path_mom[:, 1], 'b.-', label='Momentum (Damped oscillations)', linewidth=1.5)
plt.plot(path_adam[:, 0], path_adam[:, 1], 'g.-', label='Adam (Smooth, adaptive steps)', linewidth=2.0)
plt.plot(0, 0, 'r*', markersize=14, label='Global Minimum (0, 0)')
plt.xlabel('Parameter $w_1$ (Slow direction)', fontsize=11)
plt.ylabel('Parameter $w_2$ (Fast direction)', fontsize=11)
plt.title('Optimization Trajectories on Anisotropic Canyon Surface', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()